# Train/validation/test & cross-validation

Validation chooses models; the untouched test set estimates the final future error.

Train/validation/test splitting separates model selection from final evaluation. Cross-validation repeats the validation idea over folds so every example gets a turn as held-out data. Save a copy to Drive to edit.

In [ ]:
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_diabetes
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.datasets import make_regression
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.linear_model import Ridge
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...].

    All X are 2-D float feature matrices, y integer labels, so one classifier runs unchanged
    across every rung (the 'watch it scale' story). Rungs get harder: clean+separable -> real
    high-dimensional. D1 is hand-built and fully inspectable.
    """
    rungs = []

    # D1 — four hand-placed 2-D points, 2 classes, clearly separable.
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    # D2 — clean, well-separated Gaussian blobs.
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    # D3 — non-linear, overlapping two-moons with noise.
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    # D4 — real: Wine, 13 features, 3 classes.
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    # D5 — real, harder: Breast Cancer, 30 features, class imbalance.
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs

def reg_ladder():
    """D1..D5 regression ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0], [1.0], [2.0], [3.0]])
    y1 = np.array([1.0, 3.0, 5.0, 7.0])
    rungs.append(("D1 hand line y=2x+1", x1, y1))

    rng = np.random.default_rng(1)
    x2 = np.linspace(-3, 3, 120).reshape(-1, 1)
    y2 = (2.0 * x2[:, 0] + 1.0) + rng.normal(0, 0.5, size=120)
    rungs.append(("D2 linear + noise", x2, y2))

    x3 = np.linspace(-3, 3, 160).reshape(-1, 1)
    y3 = np.sin(1.5 * x3[:, 0]) + rng.normal(0, 0.2, size=160)
    rungs.append(("D3 sine (non-linear)", x3, y3))

    dia = load_diabetes()
    rungs.append(("D4 Diabetes (real, 10-D)", dia.data, dia.target))

    x5, y5 = make_regression(n_samples=300, n_features=20, n_informative=8, noise=25.0, random_state=5)
    rungs.append(("D5 high-dim + noise (20-D)", x5, y5))

    return rungs


def lesson_score(losses, cost, alternative):
    raw = float(np.sum(losses) / len(losses))
    score = raw + cost
    gap = alternative - score
    relative_gap = gap / alternative
    return raw, score, gap, relative_gap


def preview_ladder(rungs, is_regression=False):
    rows = []
    for index, item in enumerate(rungs, start=1):
        name, X, y = item
        if is_regression:
            info = f"target range {np.min(y):.2f}..{np.max(y):.2f}"
        else:
            values, counts = np.unique(y, return_counts=True)
            pairs = [f"{int(v)}:{int(c)}" for v, c in zip(values, counts)]
            info = ", ".join(pairs)
        row = {"rung": f"D{index}", "name": name, "shape": X.shape, "info": info}
        rows.append(row)
        print(row)
    name, X, y = rungs[0]
    print("sample X:")
    print(np.round(X[:5], 3))
    print("sample y:")
    print(np.round(y[:5], 3))
    return rows


def two_dimensional_view(X):
    if X.shape[1] == 1:
        return np.c_[X[:, 0], np.zeros(X.shape[0])]
    if X.shape[1] == 2:
        return X
    view = PCA(n_components=2, random_state=0).fit_transform(StandardScaler().fit_transform(X))
    return view


def stream_batches(X, y, batch_size):
    rng = np.random.default_rng(11)
    order = rng.permutation(len(y))
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        yield X[idx], y[idx]


def online_fit_predict(X, y, kind="sgd", epochs=8, batch_size=16):
    stratify = y if np.min(np.bincount(y.astype(int))) >= 2 else None
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=3,
        stratify=stratify,
    )
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    classes = np.unique(y)
    if kind == "pa":
        model = PassiveAggressiveClassifier(C=0.6, random_state=3, max_iter=1, tol=None)
    else:
        model = SGDClassifier(loss="log_loss", alpha=0.0005, random_state=3, learning_rate="optimal")
    first = True
    history = []
    batch_size = max(2, min(batch_size, len(y_train)))
    for epoch in range(epochs):
        for xb, yb in stream_batches(x_train, y_train, batch_size):
            if first:
                model.partial_fit(xb, yb, classes=classes)
                first = False
            else:
                model.partial_fit(xb, yb)
        preds = model.predict(x_test)
        history.append(float(accuracy_score(y_test, preds)))
    preds = model.predict(x_test)
    return model, scaler, x_train, x_test, y_train, y_test, preds, history


def logistic_accuracy(X, y, weighted=False):
    stratify = y if np.min(np.bincount(y.astype(int))) >= 2 else None
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=4,
        stratify=stratify,
    )
    class_weight = None
    if weighted:
        class_weight = "balanced"
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight=class_weight, random_state=4),
    )
    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    acc = float(accuracy_score(y_test, preds))
    return model, x_train, x_test, y_train, y_test, preds, acc


def expected_binary_cost(y_true, y_pred, false_negative_cost=5.0, false_positive_cost=1.0):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return float((false_negative_cost * fn + false_positive_cost * fp) / len(y_true))


def make_multi_targets(y):
    y = np.asarray(y, dtype=float)
    scale = np.std(y)
    if scale == 0:
        scale = 1.0
    centered = (y - np.mean(y)) / scale
    cuts = np.quantile(centered, [0.33, 0.66])
    ordinal = np.digitize(centered, cuts).astype(float)
    return np.c_[centered, ordinal]


def multioutput_fit_predict(X, y, alpha=1.0):
    targets = make_multi_targets(y)
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        targets,
        test_size=0.4,
        random_state=5,
    )
    model = make_pipeline(
        StandardScaler(),
        MultiOutputRegressor(Ridge(alpha=alpha)),
    )
    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    mse = float(mean_squared_error(y_test, preds))
    r2 = float(r2_score(y_test, preds, multioutput="variance_weighted"))
    return model, x_train, x_test, y_train, y_test, preds, mse, r2


def make_survival_from_classification(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    rng = np.random.default_rng(23 + X.shape[0] + X.shape[1])
    weights = np.linspace(0.4, 1.2, X.shape[1])
    linear = StandardScaler().fit_transform(X).dot(weights) / math.sqrt(X.shape[1])
    class_effect = (y == np.max(y)).astype(float) * 0.8
    risk = linear + class_effect
    event_time = np.exp(-0.45 * risk) + rng.gamma(shape=2.0, scale=0.25, size=len(y))
    censor_time = rng.gamma(shape=2.3, scale=0.5, size=len(y)) + 0.35
    observed_time = np.minimum(event_time, censor_time)
    event = (event_time <= censor_time).astype(int)
    if np.sum(event) < 3:
        event[:3] = 1
    return observed_time, event


def cox_fit(X, time, event, lr=0.03, steps=220, l2=0.02):
    X = np.asarray(X, dtype=float)
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    beta = np.zeros(X.shape[1])
    order = np.argsort(-time)
    X_desc = X[order]
    event_desc = event[order]
    for step in range(steps):
        scores = np.clip(X_desc.dot(beta), -30, 30)
        exp_scores = np.exp(scores)
        risk_sum = np.cumsum(exp_scores)
        weighted_sum = np.cumsum(exp_scores[:, None] * X_desc, axis=0)
        grad = np.zeros_like(beta)
        event_positions = np.where(event_desc == 1)[0]
        for pos in event_positions:
            grad += X_desc[pos] - weighted_sum[pos] / risk_sum[pos]
        grad = grad / max(1, len(event_positions))
        grad = grad - l2 * beta
        beta = beta + lr * grad
    return beta


def concordance_index(time, event, risk):
    total = 0
    good = 0.0
    for i in range(len(time)):
        for j in range(len(time)):
            if time[i] < time[j] and event[i] == 1:
                total += 1
                if risk[i] > risk[j]:
                    good += 1.0
                elif risk[i] == risk[j]:
                    good += 0.5
    if total == 0:
        return 0.5
    return float(good / total)


def survival_fit_score(X, y):
    time, event = make_survival_from_classification(X, y)
    stratify = y if np.min(np.bincount(y.astype(int))) >= 2 else None
    x_train, x_test, t_train, t_test, e_train, e_test = train_test_split(
        X,
        time,
        event,
        test_size=0.4,
        random_state=6,
        stratify=stratify,
    )
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    beta = cox_fit(x_train, t_train, e_train)
    train_risk = x_train.dot(beta)
    test_risk = x_test.dot(beta)
    cindex = concordance_index(t_test, e_test, test_risk)
    return beta, scaler, x_train, x_test, t_train, t_test, e_train, e_test, train_risk, test_risk, cindex


def cross_validation_gap(X, y, k=5):
    counts = np.bincount(y.astype(int))
    min_count = int(np.min(counts))
    k = max(2, min(k, min_count))
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=8),
    )
    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=8)
    result = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
    )
    train_loss = 1.0 - result["train_score"]
    val_loss = 1.0 - result["test_score"]
    gap = float(np.mean(val_loss - train_loss))
    return train_loss, val_loss, gap, cv

## The concept, built once (D1)

The lesson formula is

$$CV_K=\frac1K\sum_{k=1}^{K}R_{V_k}(\hat f_{-k})$$

Plug in the lesson losses 0.180, 0.122, and 0.437. The average is $R_S=0.739/3=0.246$, the cost is $0.050$, the score is $0.296$, and the alternative gap is $0.336-0.296=0.040$.

In [ ]:
def train_validation_test_cross_validation_method():
    losses = np.array([0.18, 0.122, 0.437], dtype=float)
    cost = 0.050
    alternative = 0.336
    fold_losses = np.array([0.18, 0.122, 0.437], dtype=float)
    cv_k = float(np.mean(fold_losses))
    raw, score, gap, relative_gap = lesson_score(losses, cost, alternative)
    assert np.isclose(cv_k, 0.246333333333)
    assert np.isclose(raw, 0.246333333333)
    assert np.isclose(score, 0.296333333333)
    assert np.isclose(gap, 0.039666666667)
    return {"fold_losses": fold_losses, "cv_k": cv_k, "raw": raw, "score": score, "gap": gap}

lesson_check = train_validation_test_cross_validation_method()
print(lesson_check)

The method returns the arithmetic pieces and asserts the exact lesson numbers before any larger data appears.

In [ ]:
assert lesson_check['score'] > lesson_check['raw']
assert lesson_check['gap'] > 0
print('lesson arithmetic locked')

## The dataset ladder

Use the shared classification ladder so the same learner runs from a hand toy to real Breast Cancer features.

In [ ]:
rungs = clf_ladder()
ladder_preview = preview_ladder(rungs, is_regression=False)

## Run the same method across D1–D5

Only the data rung changes. The metric is the plan metric for this topic.

In [ ]:
results = []
artifacts = []
for rung_index, (name, X, y) in enumerate(rungs, start=1):
    train_loss, val_loss, gap_value, cv = cross_validation_gap(X, y)
    results.append({"rung": rung_index, "name": name, "val_loss": float(np.mean(val_loss)), "gap": gap_value})
    artifacts.append((name, X, y, train_loss, val_loss, cv))
for row in results:
    print(f"D{row['rung']} validation loss={row['val_loss']:.3f}, gap={row['gap']:.3f} — {row['name']}")

## Results visualization

The first figure shows the model artifact on each rung. The second summarizes `gap` from D1 through D5.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
for index, artifact in enumerate(artifacts):
    name, X, y, train_loss, val_loss, cv = artifact
    fold_ids = np.arange(1, len(val_loss) + 1)
    axes[index].plot(fold_ids, train_loss, marker="o", label="train")
    axes[index].plot(fold_ids, val_loss, marker="o", label="validation")
    axes[index].set_title(f"D{index + 1}: CV folds")
    axes[index].set_xlabel("fold")
    axes[index].set_ylabel("loss")
    axes[index].legend(fontsize=7)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3.5))
plt.plot([row["rung"] for row in results], [row["gap"] for row in results], marker="o")
plt.axhline(0.0, color="black", linewidth=1)
plt.xticks([1, 2, 3, 4, 5], ["D1", "D2", "D3", "D4", "D5"])
plt.ylabel("validation loss - train loss")
plt.title("generalization gap vs. ladder complexity")
plt.grid(True, alpha=0.3)
plt.show()

## Pitfall on the hardest rung

The lesson warning is to optimize the raw term and forget the cost. On D5, the raw metric alone can pick a different setting than the cost-aware score.

In [ ]:
name, X, y = rungs[-1]
train_loss, val_loss, gap_value, cv = cross_validation_gap(X, y, k=5)
wrong_selection_loss = float(np.mean(train_loss))
right_selection_loss = float(np.mean(val_loss))
wrong_score = wrong_selection_loss
right_score = right_selection_loss + 0.050
print("D5 train loss only", wrong_selection_loss)
print("D5 validation loss plus lesson cost", right_score)
print("D5 fold validation losses", np.round(val_loss, 3))
print("lesson raw", 0.246, "cost", 0.050, "score", 0.296, "gap", 0.336 - 0.296)

## Evaluate it + Practice

- Compare the displayed metric with a no-skill baseline such as majority class, mean target, or random fold assignment.
- Sanity-check D1 by hand before trusting the D5 curve.
- Ablate the key idea: remove partial updates, remove PA margins, collapse outputs, ignore censoring, remove costs, or reuse the test set.
- Failure signals include unstable D5 metrics, a widening validation gap, or a cost-aware score that disagrees with the raw metric.

Practice 1: change the seed or batch/fold size and rerun the D1-to-D5 table.

Practice 2: turn off the topic-specific idea and measure the metric drop on D5.

Practice 3: add one extra diagnostic plot for the hardest rung.